In [1]:
import os
import warnings

import numpy as np
import pandas as pd
import kagglehub

# ===============================
# Instacart: Predict products in user's next order (train order)
# # - Build user/product/user-product features from PRIOR orders
# # - Create labels from TRAIN orders
# # - Train multiple gradnt-based models: LogisticRegression (SAGA), LightGBM (GBDT), and MLP (Adam)
# - Evaluate with AUC/LogLoss and order-level F1 via threshold search
# ===============================

warnings.filterwarnings("ignore")

# 1) Download data
path = kagglehub.dataset_download(
    "yasserh/instacart-online-grocery-basket-analysis-dataset"
)
print("Dataset downloaded to:", path)
BASE_PATH = path

print("\nFiles in dataset directory:")
for f in os.listdir(BASE_PATH):
    print(f)


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/bettinabopeng/miniconda3/lib/python3.12/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/Users/bettinabopeng/miniconda3/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/Users/bettinabopeng/miniconda3/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 701, in start
    self.io_loop.

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/bettinabopeng/miniconda3/lib/python3.12/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/Users/bettinabopeng/miniconda3/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/Users/bettinabopeng/miniconda3/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 701, in start
    self.io_loop.

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



Dataset downloaded to: /Users/bettinabopeng/.cache/kagglehub/datasets/yasserh/instacart-online-grocery-basket-analysis-dataset/versions/1

Files in dataset directory:
products.csv
orders.csv
order_products__train.csv
departments.csv
aisles.csv
order_products__prior.csv


In [2]:
# 2) Load CSVs
orders = pd.read_csv(os.path.join(BASE_PATH, "orders.csv"))
prior = pd.read_csv(os.path.join(BASE_PATH, "order_products__prior.csv"))
train = pd.read_csv(os.path.join(BASE_PATH, "order_products__train.csv"))

# Optional: product metadata
products = pd.read_csv(os.path.join(BASE_PATH, "products.csv"))
aisles = pd.read_csv(os.path.join(BASE_PATH, "aisles.csv"))
departments = pd.read_csv(os.path.join(BASE_PATH, "departments.csv"))

print("\nLoaded successfully:")
print("orders:", orders.shape)
print("prior:", prior.shape)
print("train:", train.shape)
print("products:", products.shape)


orders["days_since_prior_order"] = orders["days_since_prior_order"].fillna(0)


Loaded successfully:
orders: (3421083, 7)
prior: (32434489, 4)
train: (1384617, 4)
products: (49688, 4)


In [3]:
# 2b) Real-time timeline (cumulative days per user)
# Order-number recency is useful, but real-time recency in DAYS is often stronger.
orders = orders.sort_values(["user_id", "order_number"], ascending=[True, True]).copy()
orders["user_cum_days"] = orders.groupby("user_id")["days_since_prior_order"].cumsum().astype(np.float32)

# User's latest cumulative day (used to compute days since last purchase)
user_last_cum = orders.groupby("user_id")["user_cum_days"].max().rename("u_last_cum_days").reset_index()

# 3) PRIOR detail table
prior = prior.merge(
    orders[[
        "order_id",
        "user_id",
        "order_number",
        "order_dow",
        "order_hour_of_day",
        "days_since_prior_order",
        "user_cum_days",
    ]],
    on="order_id",
    how="left",
)

for col in ["order_id", "user_id", "product_id"]:
    prior[col] = pd.to_numeric(prior[col], errors="coerce")
prior = prior.dropna(subset=["order_id", "user_id", "product_id"])
prior[["order_id", "user_id", "product_id"]] = prior[["order_id", "user_id", "product_id"]].astype(np.uint32)

In [4]:
# 4) Feature engineering
user_feat = prior.groupby("user_id").agg(
    u_total_items=("product_id", "size"),
    u_unique_products=("product_id", "nunique"),
    u_total_orders=("order_number", "max"),
    u_reorder_ratio=("reordered", "mean"),
    u_avg_days=("days_since_prior_order", "mean"),
).reset_index()

# Attach user's real-time anchor day
user_feat = user_feat.merge(user_last_cum, on="user_id", how="left")
user_feat["u_last_cum_days"] = user_feat["u_last_cum_days"].fillna(0.0)

prod_feat = prior.groupby("product_id").agg(
    p_total_purchases=("user_id", "size"),
    p_unique_users=("user_id", "nunique"),
    p_reorder_ratio=("reordered", "mean"),
    p_avg_cart=("add_to_cart_order", "mean"),
).reset_index()

up_feat = prior.groupby(["user_id", "product_id"]).agg(
    up_buy_cnt=("order_id", "size"),
    up_reorder_cnt=("reordered", "sum"),
    up_cart_mean=("add_to_cart_order", "mean"),
    up_first_order=("order_number", "min"),
    up_last_order=("order_number", "max"),
).reset_index()

up_feat = up_feat.merge(user_feat[["user_id", "u_total_orders"]], on="user_id", how="left")
up_feat["up_reorder_ratio"] = up_feat["up_reorder_cnt"] / up_feat["up_buy_cnt"].replace(0, np.nan)
up_feat["up_orders_since_last"] = up_feat["u_total_orders"] - up_feat["up_last_order"]
up_feat["up_freq"] = up_feat["up_buy_cnt"] / up_feat["u_total_orders"].replace(0, np.nan)
up_feat[["up_reorder_ratio", "up_freq"]] = up_feat[["up_reorder_ratio", "up_freq"]].fillna(0.0)

In [5]:
# Real-time recency features (in days)
# Last day (cumulative) when user bought this product
up_cum = (
    prior.groupby(["user_id", "product_id"]).agg(
        up_last_cum_days=("user_cum_days", "max"),
    ).reset_index()
)
up_feat = up_feat.merge(up_cum, on=["user_id", "product_id"], how="left")

# Bring in u_last_cum_days to compute days since last purchase
up_feat = up_feat.merge(user_last_cum, on="user_id", how="left")
up_feat["u_last_cum_days"] = up_feat["u_last_cum_days"].fillna(0.0)
up_feat["up_last_cum_days"] = up_feat["up_last_cum_days"].fillna(0.0)


up_feat["up_days_since_last"] = (up_feat["u_last_cum_days"] - up_feat["up_last_cum_days"]).clip(lower=0.0)

In [6]:
# -------------------------------
# 4c) Optional: Collaborative Filtering (implicit ALS) embeddings as extra features
# -------------------------------
# Idea: learn user/product embeddings from prior interactions, then add dot/cosine scores as features.
# This is usually better as a feature for GBDT than as the final recommender.
try:
    from scipy import sparse
    import implicit

    # Build compact user/item codes using factorize (robust, guarantees 0..n-1)
    u_codes, u_uniques = pd.factorize(up_feat["user_id"], sort=False)
    p_codes, p_uniques = pd.factorize(up_feat["product_id"], sort=False)

    rows = u_codes.astype(np.int32)
    cols = p_codes.astype(np.int32)
    data = up_feat["up_buy_cnt"].astype(np.float32).values

    n_users = int(rows.max()) + 1 if len(rows) else 0
    n_items = int(cols.max()) + 1 if len(cols) else 0

    # Sanity checks (should always hold)
    if n_users <= 0 or n_items <= 0:
        raise ValueError(f"Empty ALS matrix: n_users={n_users}, n_items={n_items}")
    if rows.min() < 0 or cols.min() < 0:
        raise ValueError("Negative codes found in factorize output")

    UI = sparse.coo_matrix((data, (rows, cols)), shape=(n_users, n_items)).tocsr()

    # implicit ALS has had API conventions vary by version.
    # We'll fit on the USER×ITEM matrix (UI) and then verify/swap factor shapes if needed.

    als = implicit.als.AlternatingLeastSquares(
        factors=32,
        regularization=0.01,
        iterations=15,
        use_gpu=False,
        random_state=42,
    )

    # Confidence scaling helps; common heuristic
    alpha = 15.0
    UI_conf = (UI * alpha).astype(np.float32)

    # Fit on USER×ITEM matrix
    als.fit(UI_conf)

    user_factors = als.user_factors
    item_factors = als.item_factors

    # Robust shape reconciliation across implicit versions:
    # Desired: user_factors (n_users, k), item_factors (n_items, k)
    if user_factors.shape[0] == n_users and item_factors.shape[0] == n_items:
        pass  # OK
    elif user_factors.shape[0] == n_items and item_factors.shape[0] == n_users:
        # Some versions swap semantics; fix by swapping
        user_factors, item_factors = item_factors, user_factors
        print("[ALS] Note: swapped user_factors/item_factors to match (n_users, n_items) semantics")
    else:
        raise ValueError(
            f"ALS factor shapes unexpected: user_factors={user_factors.shape}, item_factors={item_factors.shape}; "
            f"expected (n_users={n_users}, n_items={n_items})"
        )

    # Sanity: max codes must be within factor matrix bounds
    if rows.max() >= user_factors.shape[0] or cols.max() >= item_factors.shape[0]:
        raise ValueError(
            f"ALS indexing would be OOB: rows.max={int(rows.max())}, users={user_factors.shape[0]}; "
            f"cols.max={int(cols.max())}, items={item_factors.shape[0]}"
        )

    uf = user_factors[rows]
    vf = item_factors[cols]

    # Dot product similarity
    up_feat["als_dot"] = np.sum(uf * vf, axis=1).astype(np.float32)

    # Cosine similarity
    uf_norm = np.linalg.norm(uf, axis=1)
    vf_norm = np.linalg.norm(vf, axis=1)
    denom = np.maximum(uf_norm * vf_norm, 1e-8)
    up_feat["als_cos"] = (up_feat["als_dot"].values / denom).astype(np.float32)

    print("\n[ALS] Added embedding features: als_dot, als_cos")

except Exception as e:
    # If scipy/implicit not installed, skip safely.
    print("\n[ALS] Skipped (implicit/scipy not installed or failed). Error:")
    print(e)
    print("[ALS] Debug: als_dot/als_cos set to 0.0")
    up_feat["als_dot"] = 0.0
    up_feat["als_cos"] = 0.0

  0%|          | 0/15 [00:00<?, ?it/s]


[ALS] Added embedding features: als_dot, als_cos


In [7]:
# 4b) Simple baseline: per-user Top-11 most frequent PRIOR products
# Build a dict: user_id -> set(top11 product_id)
# Deterministic tie-break: higher up_buy_cnt first, then smaller product_id
up_rank = up_feat[["user_id", "product_id", "up_buy_cnt"]].copy()
up_rank = up_rank.sort_values(["user_id", "up_buy_cnt", "product_id"], ascending=[True, False, True])
user_top11 = (
    up_rank.groupby("user_id")
    .head(11)
    .groupby("user_id")["product_id"]
    .apply(set)
    .to_dict()
)

In [8]:
# 5) Labels from TRAIN orders

train_orders = orders.loc[orders.eval_set == "train", ["order_id", "user_id"]].copy()
train_orders["order_id"] = train_orders["order_id"].astype(np.uint32)
train_orders["user_id"] = train_orders["user_id"].astype(np.uint32)

train = train.merge(train_orders, on="order_id", how="left")
train = train.dropna(subset=["user_id", "product_id"])
train["user_id"] = pd.to_numeric(train["user_id"], errors="coerce").astype(np.uint32)
train["product_id"] = pd.to_numeric(train["product_id"], errors="coerce").astype(np.uint32)
train["label"] = 1

y_map = train[["user_id", "product_id", "label"]].drop_duplicates()

full = up_feat.merge(y_map, on=["user_id", "product_id"], how="left")
full["label"] = full["label"].fillna(0).astype(np.uint8)

# Merge user/product features (avoid duplicating u_total_orders)
user_feat_for_merge = user_feat.drop(columns=["u_total_orders"], errors="ignore")
full = full.merge(user_feat_for_merge, on="user_id", how="left")
full = full.merge(prod_feat, on="product_id", how="left")

# Reconcile duplicated u_last_cum_days columns (can appear as _x/_y depending on merge order)
if "u_last_cum_days" not in full.columns:
    if "u_last_cum_days_x" in full.columns:
        full["u_last_cum_days"] = full["u_last_cum_days_x"]
    elif "u_last_cum_days_y" in full.columns:
        full["u_last_cum_days"] = full["u_last_cum_days_y"]

for c in ["u_last_cum_days_x", "u_last_cum_days_y"]:
    if c in full.columns:
        full = full.drop(columns=[c])

# Safety fill
if "u_last_cum_days" in full.columns:
    full["u_last_cum_days"] = full["u_last_cum_days"].fillna(0.0)

if "u_total_orders" not in full.columns:
    if "u_total_orders_x" in full.columns:
        full["u_total_orders"] = full["u_total_orders_x"]
    elif "u_total_orders_y" in full.columns:
        full["u_total_orders"] = full["u_total_orders_y"]
for c in ["u_total_orders_x", "u_total_orders_y"]:
    if c in full.columns:
        full = full.drop(columns=[c])

# Guard: ensure u_last_cum_days exists
if "u_last_cum_days" not in full.columns:
    full = full.merge(user_last_cum, on="user_id", how="left")
    full["u_last_cum_days"] = full["u_last_cum_days"].fillna(0.0)

full = full.merge(products[["product_id", "aisle_id", "department_id"]], on="product_id", how="left")
full["aisle_id"] = pd.to_numeric(full["aisle_id"], errors="coerce").fillna(0).astype(np.uint32)
full["department_id"] = pd.to_numeric(full["department_id"], errors="coerce").fillna(0).astype(np.uint32)

print("\nTraining table shape (candidates):", full.shape)
print("Label positive rate:", full["label"].mean())


Training table shape (candidates): (13307953, 27)
Label positive rate: 0.062280352207435656


In [9]:
# 6) Split by user

from sklearn.model_selection import train_test_split
unique_users = full["user_id"].unique()
train_users, val_users = train_test_split(unique_users, test_size=0.2, random_state=42)

train_df = full[full.user_id.isin(train_users)].copy()
val_df = full[full.user_id.isin(val_users)].copy()

features = [
    "u_total_items",
    "u_unique_products",
    "u_total_orders",
    "u_reorder_ratio",
    "u_avg_days",
    "u_last_cum_days",
    "p_total_purchases",
    "p_unique_users",
    "p_reorder_ratio",
    "p_avg_cart",
    "up_buy_cnt",
    "up_reorder_cnt",
    "up_cart_mean",
    "up_first_order",
    "up_last_order",
    "up_reorder_ratio",
    "up_orders_since_last",
    "up_days_since_last",
    "up_freq",
    "als_dot",
    "als_cos",
    "aisle_id",
    "department_id",
]

missing = [c for c in features if c not in train_df.columns]
if missing:
    raise KeyError(f"Missing feature columns in train_df: {missing}. Available columns: {list(train_df.columns)}")


train_df[features] = train_df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
val_df[features] = val_df[features].replace([np.inf, -np.inf], np.nan).fillna(0)

X_train = train_df[features]
y_train = train_df["label"].values
X_val = val_df[features]
y_val = val_df["label"].values

In [10]:
# 7) Logistic Regression (gradient)

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, log_loss, average_precision_score, brier_score_loss

scaler = StandardScaler()
Xtr_s = scaler.fit_transform(X_train)
Xva_s = scaler.transform(X_val)

lr = LogisticRegression(solver="saga", max_iter=200, n_jobs=-1, class_weight="balanced")
lr.fit(Xtr_s, y_train)

p_lr = lr.predict_proba(Xva_s)[:, 1]
print("\n[LogReg] AUC:", roc_auc_score(y_val, p_lr))
print("[LogReg] LogLoss:", log_loss(y_val, p_lr))
print("[LogReg] AP (PR-AUC):", average_precision_score(y_val, p_lr))
print("[LogReg] Brier:", brier_score_loss(y_val, p_lr))


[LogReg] AUC: 0.8114221671501041
[LogReg] LogLoss: 0.5317642819444536
[LogReg] AP (PR-AUC): 0.24652564438993868
[LogReg] Brier: 0.17929125999202505


In [11]:
# -------------------------------
# 8) LightGBM (GBDT, gradient boosting)
# -------------------------------
try:
    import lightgbm as lgb

    # Tell LightGBM to treat taxonomy ids as categorical features
    cat_feats = ["aisle_id", "department_id"]
    lgb_train = lgb.Dataset(X_train, label=y_train, categorical_feature=cat_feats, free_raw_data=False)
    lgb_val = lgb.Dataset(X_val, label=y_val, reference=lgb_train, categorical_feature=cat_feats, free_raw_data=False)

    # Class imbalance handling: scale_pos_weight ~= (#neg / #pos)
    pos_rate = float(np.mean(y_train))
    scale_pos_weight = (1.0 - pos_rate) / max(pos_rate, 1e-6)

    params = dict(
        objective="binary",
        learning_rate=0.05,
        num_leaves=64,
        min_data_in_leaf=200,
        feature_fraction=0.8,
        bagging_fraction=0.8,
        bagging_freq=1,
        lambda_l2=1.0,
        lambda_l1=0.0,
        scale_pos_weight=scale_pos_weight,
        metric=["auc", "binary_logloss"],
    )

    gbm = lgb.train(
        params,
        lgb_train,
        valid_sets=[lgb_train, lgb_val],
        num_boost_round=3000,
        callbacks=[lgb.early_stopping(50)],
    )

    p_lgb = gbm.predict(X_val, num_iteration=gbm.best_iteration)
    print("\n[LightGBM] AUC:", roc_auc_score(y_val, p_lgb))
    print("[LightGBM] LogLoss:", log_loss(y_val, p_lgb))

    fi = pd.DataFrame({
        "feature": gbm.feature_name(),
        "importance": gbm.feature_importance(importance_type="gain"),
    }).sort_values("importance", ascending=False)
    print("\nTop 20 feature importances (gain):")
    print(fi.head(20).to_string(index=False))
    # Quick additional metrics (row-level)
    print("[LightGBM] AP (PR-AUC):", average_precision_score(y_val, p_lgb))
    print("[LightGBM] Brier:", brier_score_loss(y_val, p_lgb))

except Exception as e:
    print("\n[LightGBM] Skipped (not installed or failed to import/train). Error:")
    print(e)
    p_lgb = None

[LightGBM] [Info] Number of positive: 663178, number of negative: 9964588
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.116281 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4276
[LightGBM] [Info] Number of data points in the train set: 10627766, number of used features: 23
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.062401 -> initscore=-2.709749
[LightGBM] [Info] Start training from score -2.709749
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[2]	training's auc: 0.81308	training's binary_logloss: 0.222753	valid_1's auc: 0.811495	valid_1's binary_logloss: 0.221628

[LightGBM] AUC: 0.8114948657144787
[LightGBM] LogLoss: 0.22162827542052158

Top 20 feature importances (gain):
             feature   importance
up_orders_since_last 1.687308e+07
          up_buy_cnt 3.816828e+06
   

In [12]:
# # -------------------------------
# # 8b) XGBoost (GBDT) - strong baseline for tabular features
# # -------------------------------
# try:
#     import xgboost as xgb
#
#     dtrain = xgb.DMatrix(X_train, label=y_train)
#     dval = xgb.DMatrix(X_val, label=y_val)
#
#     pos_rate = float(np.mean(y_train))
#     scale_pos_weight = (1.0 - pos_rate) / max(pos_rate, 1e-6)
#
#     xgb_params = {
#         "objective": "binary:logistic",
#         "eval_metric": ["auc", "logloss"],
#         "eta": 0.05,
#         "max_depth": 6,
#         "min_child_weight": 5,
#         "subsample": 0.8,
#         "colsample_bytree": 0.8,
#         "lambda": 1.0,
#         "alpha": 0.0,
#         "scale_pos_weight": scale_pos_weight,
#         "tree_method": "hist",
#         "seed": 42,
#     }
#
#     xgb_model = xgb.train(
#         xgb_params,
#         dtrain,
#         num_boost_round=5000,
#         evals=[(dtrain, "train"), (dval, "val")],
#         early_stopping_rounds=50,
#         verbose_eval=False,
#     )
#
#     p_xgb = xgb_model.predict(dval, iteration_range=(0, xgb_model.best_iteration + 1))
#     print("\n[XGBoost] AUC:", roc_auc_score(y_val, p_xgb))
#     print("[XGBoost] LogLoss:", log_loss(y_val, p_xgb))
#     print("[XGBoost] AP (PR-AUC):", average_precision_score(y_val, p_xgb))
#     print("[XGBoost] Brier:", brier_score_loss(y_val, p_xgb))
#
# except Exception as e:
#     print("\n[XGBoost] Skipped (not installed or failed). Error:")
#     print(e)
#     p_xgb = None


In [13]:
# 8c) CatBoost (GBDT) - handles categorical features well

try:
    from catboost import CatBoostClassifier, Pool

    # CatBoost wants categorical feature indices
    cat_cols = ["aisle_id", "department_id"]
    cat_idx = [X_train.columns.get_loc(c) for c in cat_cols if c in X_train.columns]

    train_pool = Pool(X_train, label=y_train, cat_features=cat_idx)
    val_pool = Pool(X_val, label=y_val, cat_features=cat_idx)

    pos_rate = float(np.mean(y_train))
    # Class weights: [neg, pos]
    class_weights = [1.0, (1.0 - pos_rate) / max(pos_rate, 1e-6)]

    cb = CatBoostClassifier(
        loss_function="Logloss",
        eval_metric="AUC",
        learning_rate=0.05,
        depth=8,
        l2_leaf_reg=3.0,
        random_seed=42,
        iterations=5000,
        od_type="Iter",
        od_wait=50,
        verbose=False,
        class_weights=class_weights,
    )

    cb.fit(train_pool, eval_set=val_pool, use_best_model=True)

    p_cat = cb.predict_proba(val_pool)[:, 1]
    print("\n[CatBoost] AUC:", roc_auc_score(y_val, p_cat))
    print("[CatBoost] LogLoss:", log_loss(y_val, p_cat))
    print("[CatBoost] AP (PR-AUC):", average_precision_score(y_val, p_cat))
    print("[CatBoost] Brier:", brier_score_loss(y_val, p_cat))

except Exception as e:
    print("\n[CatBoost] Skipped (not installed or failed). Error:")
    print(e)
    p_cat = None


[CatBoost] AUC: 0.8218511718224666
[CatBoost] LogLoss: 0.514098573873096
[CatBoost] AP (PR-AUC): 0.26715281663860374
[CatBoost] Brier: 0.17185834173369027


In [14]:
# 9) MLP (Adam, gradient)
from sklearn.neural_network import MLPClassifier

mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation="relu",
    solver="adam",
    alpha=1e-4,
    batch_size=4096,
    learning_rate_init=1e-3,
    max_iter=10,
    random_state=42,
)
mlp.fit(Xtr_s, y_train)
p_mlp = mlp.predict_proba(Xva_s)[:, 1]
print("\n[MLP] AUC:", roc_auc_score(y_val, p_mlp))
print("[MLP] LogLoss:", log_loss(y_val, p_mlp))
print("[MLP] AP (PR-AUC):", average_precision_score(y_val, p_mlp))
print("[MLP] Brier:", brier_score_loss(y_val, p_mlp))


[MLP] AUC: 0.8199807704475921
[MLP] LogLoss: 0.1884711625266385
[MLP] AP (PR-AUC): 0.261459484022873
[MLP] Brier: 0.051091274462190084


In [15]:
# 10) Order-level F1 threshold search
user_to_train_order = train_orders.set_index("user_id")["order_id"].to_dict()
true_by_order = train.groupby("order_id")["product_id"].apply(set).to_dict()
val_order_ids = set(train_orders.loc[train_orders.user_id.isin(val_users), "order_id"].values.tolist())

#
# Build a registry of row-level probability outputs for fair Top-K order-level comparison
model_probs = {"LogReg": p_lr, "MLP": p_mlp}
if "p_lgb" in globals() and p_lgb is not None:
    model_probs["LightGBM"] = p_lgb
if "p_xgb" in globals() and p_xgb is not None:
    model_probs["XGBoost"] = p_xgb
if "p_cat" in globals() and p_cat is not None:
    model_probs["CatBoost"] = p_cat

# Prefer stronger GBDT models for ranking/calibration diagnostics shown below
if "CatBoost" in model_probs:
    prob_used = model_probs["CatBoost"]
    prob_name = "CatBoost"
elif "XGBoost" in model_probs:
    prob_used = model_probs["XGBoost"]
    prob_name = "XGBoost"
elif "LightGBM" in model_probs:
    prob_used = model_probs["LightGBM"]
    prob_name = "LightGBM"
else:
    prob_used = model_probs["LogReg"]
    prob_name = "LogReg"

val_pred = val_df[["user_id", "product_id"]].copy()
val_pred["order_id"] = val_pred["user_id"].map(user_to_train_order)

# Align probs to val_df index so later filtering won't break lengths
prob_used_s = pd.Series(prob_used, index=val_df.index)
val_pred["prob"] = prob_used_s.loc[val_pred.index].values

val_pred = val_pred.dropna(subset=["order_id"]).copy()
val_pred["order_id"] = val_pred["order_id"].astype(np.uint32)

In [16]:
# -------- Order-level evaluation helpers --------
true_size_by_order = {oid: len(s) for oid, s in true_by_order.items()}


def prf_for_order(pred_set, true_set):
    # Precision/Recall/F1 on sets
    if len(pred_set) == 0 and len(true_set) == 0:
        return 1.0, 1.0, 1.0
    if len(pred_set) == 0:
        return 0.0, 0.0, 0.0
    if len(true_set) == 0:
        return 0.0, 0.0, 0.0
    inter = len(pred_set & true_set)
    p = inter / len(pred_set) if len(pred_set) else 0.0
    r = inter / len(true_set) if len(true_set) else 0.0
    f1 = (2.0 * p * r / (p + r)) if (p + r) > 0 else 0.0
    return p, r, f1


def eval_threshold_prf(df, th):
    # Build predicted sets
    pred_by_order = (
        df[df.prob >= th]
        .groupby("order_id")["product_id"]
        .apply(set)
        .to_dict()
    )

    ps, rs, f1s = [], [], []
    pred_sizes, true_sizes = [], []

    for oid in val_order_ids:
        true_set = true_by_order.get(oid, set())
        pred_set = pred_by_order.get(oid, set())
        p, r, f1 = prf_for_order(pred_set, true_set)
        ps.append(p)
        rs.append(r)
        f1s.append(f1)
        pred_sizes.append(len(pred_set))
        true_sizes.append(len(true_set))

    out = {
        "precision": float(np.mean(ps)) if ps else 0.0,
        "recall": float(np.mean(rs)) if rs else 0.0,
        "f1": float(np.mean(f1s)) if f1s else 0.0,
        "pred_size_mean": float(np.mean(pred_sizes)) if pred_sizes else 0.0,
        "pred_size_p50": float(np.percentile(pred_sizes, 50)) if pred_sizes else 0.0,
        "pred_size_p90": float(np.percentile(pred_sizes, 90)) if pred_sizes else 0.0,
        "true_size_mean": float(np.mean(true_sizes)) if true_sizes else 0.0,
        "true_size_p50": float(np.percentile(true_sizes, 50)) if true_sizes else 0.0,
        "true_size_p90": float(np.percentile(true_sizes, 90)) if true_sizes else 0.0,
    }
    return out


In [17]:
def eval_recall_at_k(df, ks=(10, 20, 30)):
    # Ranking-based evaluation independent of a global threshold
    # For each order, take top-K products by prob and compute recall@K and hit@K.
    df2 = df[["order_id", "product_id", "prob"]].copy()
    df2 = df2.sort_values(["order_id", "prob"], ascending=[True, False])

    out = {}
    for k in ks:
        topk = df2.groupby("order_id").head(k)
        pred_by_order = topk.groupby("order_id")["product_id"].apply(set).to_dict()

        recalls = []
        hits = []
        for oid in val_order_ids:
            true_set = true_by_order.get(oid, set())
            true_n = len(true_set)
            if true_n == 0:
                continue
            pred_set = pred_by_order.get(oid, set())
            inter = len(pred_set & true_set)
            recalls.append(inter / true_n)
            hits.append(1.0 if inter > 0 else 0.0)

        out[f"recall@{k}"] = float(np.mean(recalls)) if recalls else 0.0
        out[f"hit@{k}"] = float(np.mean(hits)) if hits else 0.0

    return out

In [18]:
# -- Fixed Top-K order-level P/R/F1 evaluation for fair comparison to baseline --
def eval_topk_prf(df, k=11):
    # For each order, take top-k products by prob and compute order-level P/R/F1 and size stats.
    df2 = df[["order_id", "product_id", "prob"]].copy()
    df2 = df2.sort_values(["order_id", "prob", "product_id"], ascending=[True, False, True])
    topk = df2.groupby("order_id").head(k)
    pred_by_order = topk.groupby("order_id")["product_id"].apply(set).to_dict()

    ps, rs, f1s = [], [], []
    pred_sizes, true_sizes = [], []

    for oid in val_order_ids:
        true_set = true_by_order.get(oid, set())
        pred_set = pred_by_order.get(oid, set())
        p, r, f1 = prf_for_order(pred_set, true_set)
        ps.append(p)
        rs.append(r)
        f1s.append(f1)
        pred_sizes.append(len(pred_set))
        true_sizes.append(len(true_set))

    out = {
        "k": int(k),
        "precision": float(np.mean(ps)) if ps else 0.0,
        "recall": float(np.mean(rs)) if rs else 0.0,
        "f1": float(np.mean(f1s)) if f1s else 0.0,
        "pred_size_mean": float(np.mean(pred_sizes)) if pred_sizes else 0.0,
        "pred_size_p50": float(np.percentile(pred_sizes, 50)) if pred_sizes else 0.0,
        "pred_size_p90": float(np.percentile(pred_sizes, 90)) if pred_sizes else 0.0,
        "true_size_mean": float(np.mean(true_sizes)) if true_sizes else 0.0,
        "true_size_p50": float(np.percentile(true_sizes, 50)) if true_sizes else 0.0,
        "true_size_p90": float(np.percentile(true_sizes, 90)) if true_sizes else 0.0,
    }
    return out

In [19]:
# -- Fixed Top-K order-level P/R/F1 evaluation for fair comparison to baseline --
def eval_topk_prf(df, k=11):
    # For each order, take top-k products by prob and compute order-level P/R/F1 and size stats.
    df2 = df[["order_id", "product_id", "prob"]].copy()
    df2 = df2.sort_values(["order_id", "prob", "product_id"], ascending=[True, False, True])
    topk = df2.groupby("order_id").head(k)
    pred_by_order = topk.groupby("order_id")["product_id"].apply(set).to_dict()

    ps, rs, f1s = [], [], []
    pred_sizes, true_sizes = [], []

    for oid in val_order_ids:
        true_set = true_by_order.get(oid, set())
        pred_set = pred_by_order.get(oid, set())
        p, r, f1 = prf_for_order(pred_set, true_set)
        ps.append(p)
        rs.append(r)
        f1s.append(f1)
        pred_sizes.append(len(pred_set))
        true_sizes.append(len(true_set))

    out = {
        "k": int(k),
        "precision": float(np.mean(ps)) if ps else 0.0,
        "recall": float(np.mean(rs)) if rs else 0.0,
        "f1": float(np.mean(f1s)) if f1s else 0.0,
        "pred_size_mean": float(np.mean(pred_sizes)) if pred_sizes else 0.0,
        "pred_size_p50": float(np.percentile(pred_sizes, 50)) if pred_sizes else 0.0,
        "pred_size_p90": float(np.percentile(pred_sizes, 90)) if pred_sizes else 0.0,
        "true_size_mean": float(np.mean(true_sizes)) if true_sizes else 0.0,
        "true_size_p50": float(np.percentile(true_sizes, 50)) if true_sizes else 0.0,
        "true_size_p90": float(np.percentile(true_sizes, 90)) if true_sizes else 0.0,
    }
    return out


def build_val_pred_with_probs(prob_array):
    """Build order-aligned validation prediction rows for a given probability vector aligned to val_df."""
    tmp = val_df[["user_id", "product_id"]].copy()
    tmp["order_id"] = tmp["user_id"].map(user_to_train_order)
    prob_s = pd.Series(prob_array, index=val_df.index)
    tmp["prob"] = prob_s.loc[tmp.index].values
    tmp = tmp.dropna(subset=["order_id"]).copy()
    tmp["order_id"] = tmp["order_id"].astype(np.uint32)
    return tmp


def print_topk_result(name, res, k):
    print(
        f"  {name:<10} Top-{k} -> F1={res['f1']:.5f} (P={res['precision']:.5f}, R={res['recall']:.5f}) "
        f"| pred_size_mean={res['pred_size_mean']:.2f}"
    )

In [20]:
def expected_calibration_error(y_true, y_prob, n_bins=15):
    # Simple ECE with equal-width bins on [0,1]
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        m = (y_prob >= lo) & (y_prob < hi) if i < n_bins - 1 else (y_prob >= lo) & (y_prob <= hi)
        if not np.any(m):
            continue
        acc = np.mean(y_true[m])
        conf = np.mean(y_prob[m])
        w = np.mean(m)
        ece += w * abs(acc - conf)
    return float(ece)

In [21]:
TOPK_EVAL_K = 11
print(f"\nOrder-level Top-{TOPK_EVAL_K} evaluation (same rule for all models; fair vs baseline)")

order_topk_results = {}
val_pred_by_model = {}

# Print in a stable, readable order
model_print_order = ["LogReg", "LightGBM", "XGBoost", "CatBoost", "MLP"]
for mname in model_print_order:
    if mname not in model_probs:
        continue
    vpm = build_val_pred_with_probs(model_probs[mname])
    val_pred_by_model[mname] = vpm
    res = eval_topk_prf(vpm, k=TOPK_EVAL_K)
    order_topk_results[mname] = res
    print_topk_result(mname, res, TOPK_EVAL_K)

# Keep a detailed print for the chosen model (used below for ranking/calibration diagnostics)
model_topk = order_topk_results.get(prob_name, eval_topk_prf(val_pred, k=TOPK_EVAL_K))
print(
    f"  [Chosen for diagnostics: {prob_name}] true_size_mean={model_topk['true_size_mean']:.2f} "
    f"(p50={model_topk['true_size_p50']:.0f}, p90={model_topk['true_size_p90']:.0f})"
)

# Ranking metrics (threshold-free)
rank_metrics = eval_recall_at_k(val_pred, ks=(10, 20, 30))
print("\nRanking metrics (threshold-free):")
for k, v in rank_metrics.items():
    print(f"  {k}: {v:.5f}")

# Row-level probability quality summary for the chosen model
# (AP, Brier were already printed for each model; ECE is added here.)
# Build y_true / y_prob aligned with val_df
if prob_name == "CatBoost":
    y_prob = p_cat
elif prob_name == "XGBoost":
    y_prob = p_xgb
elif prob_name == "LightGBM":
    y_prob = p_lgb
elif prob_name == "LogReg":
    y_prob = p_lr
else:
    y_prob = prob_used
print("\nCalibration (row-level):")
print("  ECE (15 bins):", expected_calibration_error(y_val, y_prob, n_bins=15))


Order-level Top-11 evaluation (same rule for all models; fair vs baseline)
  LogReg     Top-11 -> F1=0.28245 (P=0.28936, R=0.36143) | pred_size_mean=10.68
  LightGBM   Top-11 -> F1=0.28387 (P=0.29051, R=0.36341) | pred_size_mean=10.68
  CatBoost   Top-11 -> F1=0.29129 (P=0.29872, R=0.37173) | pred_size_mean=10.68
  MLP        Top-11 -> F1=0.28991 (P=0.29719, R=0.37006) | pred_size_mean=10.68
  [Chosen for diagnostics: CatBoost] true_size_mean=10.55 (p50=9, p90=21)

Ranking metrics (threshold-free):
  recall@10: 0.35662
  hit@10: 0.87418
  recall@20: 0.46263
  hit@20: 0.91057
  recall@30: 0.51543
  hit@30: 0.92280

Calibration (row-level):
  ECE (15 bins): 0.298252107844462


In [22]:
# 10b) Baseline evaluation: predict each user's PRIOR top-11 frequent products
# -------------------------------
print("\nBaseline: per-user PRIOR top-11 frequent products")

# Create reverse map order_id -> user_id for validation orders
order_to_user = train_orders.set_index("order_id")["user_id"].to_dict()
baseline_pred_by_order = {}
for oid in val_order_ids:
    uid = order_to_user.get(oid)
    baseline_pred_by_order[oid] = user_top11.get(uid, set())

# Compute order-level P/R/F1 and size stats
b_ps, b_rs, b_f1s = [], [], []
b_pred_sizes, b_true_sizes = [], []
for oid in val_order_ids:
    true_set = true_by_order.get(oid, set())
    pred_set = baseline_pred_by_order.get(oid, set())
    p, r, f1 = prf_for_order(pred_set, true_set)
    b_ps.append(p)
    b_rs.append(r)
    b_f1s.append(f1)
    b_pred_sizes.append(len(pred_set))
    b_true_sizes.append(len(true_set))

b_precision = float(np.mean(b_ps)) if b_ps else 0.0
b_recall = float(np.mean(b_rs)) if b_rs else 0.0
b_f1 = float(np.mean(b_f1s)) if b_f1s else 0.0

print(
    f"  P={b_precision:.5f}, R={b_recall:.5f}, F1={b_f1:.5f} | "
    f"pred_size_mean={np.mean(b_pred_sizes):.2f} (p50={np.percentile(b_pred_sizes,50):.0f}, p90={np.percentile(b_pred_sizes,90):.0f}) "
    f"vs true_size_mean={np.mean(b_true_sizes):.2f}"
)

# Also compute recall@K / hit@K for the baseline using the same predicted sets
# For baseline, top-K is effectively K=11 (no ranking within the set beyond membership)
# We'll report recall@11 and hit@11.
recalls = []
hits = []
for oid in val_order_ids:
    true_set = true_by_order.get(oid, set())
    true_n = len(true_set)
    if true_n == 0:
        continue
    pred_set = baseline_pred_by_order.get(oid, set())
    inter = len(pred_set & true_set)
    recalls.append(inter / true_n)
    hits.append(1.0 if inter > 0 else 0.0)

print(f"  recall@11: {float(np.mean(recalls)) if recalls else 0.0:.5f}")
print(f"  hit@11: {float(np.mean(hits)) if hits else 0.0:.5f}")


Baseline: per-user PRIOR top-11 frequent products
  P=0.26442, R=0.33205, F1=0.25827 | pred_size_mean=10.68 (p50=11, p90=11) vs true_size_mean=10.55
  recall@11: 0.33205
  hit@11: 0.85505


In [23]:
#
# 11) Simple ensemble (optional) - FIXED alignment
# -------------------------------
# Choose best available tree model for ensembling
# p_tree = None
# if "p_cat" in globals() and p_cat is not None:
#     p_tree = p_cat
# elif "p_xgb" in globals() and p_xgb is not None:
#     p_tree = p_xgb
# elif p_lgb is not None:
#     p_tree = p_lgb
#
# if p_tree is not None:
#     p_ens = 0.2 * p_lr + 0.8 * p_tree
#     p_ens_s = pd.Series(p_ens, index=val_df.index)
#
#     val_pred_ens = val_df[["user_id", "product_id"]].copy()
#     val_pred_ens["order_id"] = val_pred_ens["user_id"].map(user_to_train_order)
#     val_pred_ens["prob"] = p_ens_s.loc[val_pred_ens.index].values
#     val_pred_ens = val_pred_ens.dropna(subset=["order_id"]).copy()
#     val_pred_ens["order_id"] = val_pred_ens["order_id"].astype(np.uint32)
#
#     print(f"\nOrder-level Top-{TOPK_EVAL_K} evaluation using Ensemble (0.2*LR + 0.8*Tree):")
#     ens_topk = eval_topk_prf(val_pred_ens, k=TOPK_EVAL_K)
#     order_topk_results["Ensemble"] = ens_topk
#     print(
#         f"  Top-{TOPK_EVAL_K} -> F1={ens_topk['f1']:.5f} (P={ens_topk['precision']:.5f}, R={ens_topk['recall']:.5f}) "
#         f"| pred_size_mean={ens_topk['pred_size_mean']:.2f} (p50={ens_topk['pred_size_p50']:.0f}, p90={ens_topk['pred_size_p90']:.0f})"
#     )
#     print(
#         f"  true_size_mean={ens_topk['true_size_mean']:.2f} (p50={ens_topk['true_size_p50']:.0f}, p90={ens_topk['true_size_p90']:.0f})"
#     )
#
#     if "CatBoost" in order_topk_results:
#         delta_f1 = order_topk_results["Ensemble"]["f1"] - order_topk_results["CatBoost"]["f1"]
#         print(f"  [Ensemble vs CatBoost] ΔF1={delta_f1:+.5f}")
#
#     ens_rank = eval_recall_at_k(val_pred_ens, ks=(10, 20, 30))
#     print("\n[Ensemble] Ranking metrics (threshold-free):")
#     for k, v in ens_rank.items():
#         print(f"  {k}: {v:.5f}")

print("\n[Ensemble] Skipped (manually disabled).")



[Ensemble] Skipped (manually disabled).


In [24]:
# Print unified summary table of Top-11 order-level results for all models + baseline
# (moved here to ensure Ensemble is included)
# -------------------------------
summary_rows = []
for mname in ["LogReg", "LightGBM", "XGBoost", "CatBoost", "MLP", "Ensemble"]:
    if mname in order_topk_results:
        r = order_topk_results[mname]
        summary_rows.append({
            "method": f"{mname} Top-{TOPK_EVAL_K}",
            "precision": r["precision"],
            "recall": r["recall"],
            "f1": r["f1"],
            "pred_size_mean": r["pred_size_mean"],
        })

summary_rows.append({
    "method": f"Baseline Top-{TOPK_EVAL_K}",
    "precision": b_precision,
    "recall": b_recall,
    "f1": b_f1,
    "pred_size_mean": float(np.mean(b_pred_sizes)) if len(b_pred_sizes) else 0.0,
})

summary_df = pd.DataFrame(summary_rows).sort_values("f1", ascending=False)
print(f"\nOrder-level Top-{TOPK_EVAL_K} summary (fair comparison)")
print(summary_df.to_string(index=False, float_format=lambda x: f"{x:.5f}"))

print("\nDone.")



Order-level Top-11 summary (fair comparison)
         method  precision  recall      f1  pred_size_mean
CatBoost Top-11    0.29872 0.37173 0.29129        10.67527
     MLP Top-11    0.29719 0.37006 0.28991        10.67527
LightGBM Top-11    0.29051 0.36341 0.28387        10.67527
  LogReg Top-11    0.28936 0.36143 0.28245        10.67527
Baseline Top-11    0.26442 0.33205 0.25827        10.67527

Done.
